# NFDI4Objects KG — Ogham Sites on a Map (local variant)

## About this notebook

This notebook takes the same
[NFDI4Objects Knowledge Graph](https://graph.nfdi4objects.net/)
dataset that the companion notebook `n4okg-ogham-sites-county.ipynb`
visualises as counts, and plots each individual
[Ogham site](http://ontology.ogham.link/OghamSite) on two interactive
[Leaflet](https://leafletjs.com/) maps (via
[`folium`](https://python-visualization.github.io/folium/)). The
first map shows individual sites as county-coloured markers with
pop-up details; the second shows the same points styled by stone
count — marker size and colour both encode how many stones were
disclosed at each site.

It is part of an Open Educational Resource series on knowledge
graphs and linked open data. A browser-executable variant of this
notebook is available as `n4okg-ogham-sites-map-live.qmd` (same
content, runs in the browser via Pyodide without any local Python
setup).

### Why this dataset?

Ogham sites from the N4O KG's
[`collection/9`](https://graph.nfdi4objects.net/collection/9) are a
good fit for a map teaching example because:

- Each site has a **GeoSPARQL `hasGeometry`/`asWKT` literal** with
  explicit WKT point coordinates — a common, well-specified
  encoding that generalises to many other knowledge graphs.
- The **stone-count-per-site** metadata is rich enough for two
  different map treatments: a categorical view coloured by county,
  and a continuous view sized and coloured by stone count.
- The sites concentrate in just two Irish regions, so the map is
  geographically interesting without being sparse.

### Data-context notes

Two subtleties about the coordinates that we handle here and that
generalise to other GeoSPARQL-using KGs:

- **WKT is `lon lat`, Leaflet is `lat, lon`.** Getting this the
  wrong way round is one of the most common mistakes when plotting
  spatial data. Our parser swaps the order explicitly.
- **Some endpoints uppercase the `POINT` keyword, some don't.**
  Wikidata writes `Point(...)`, the N4O KG writes `POINT(...)`.
  Our parser is case-insensitive, which is a reasonable defensive
  posture when working across endpoints.

A third point worth mentioning: N4O KG data is contributed by many
projects under shared infrastructure, each with its own ontology.
The Ogham ontology (`oghamonto:`) is separate from the NFDI4Objects
core vocabulary. This is normal for domain-rich knowledge graphs
and one of the reasons SPARQL is expressive: you can mix
vocabularies freely in a single query.

### Requirements

```bash
pip install SPARQLWrapper pandas folium
```

### Tooling notes

- **`SPARQLWrapper`** to query the N4O KG. The browser-executable
  variant uses `pyodide.http.pyfetch` instead.
- **`folium`** to build the interactive map. `folium` is a Python
  wrapper around Leaflet: you describe the map in Python, and the
  library emits the HTML + JavaScript Jupyter needs to render it
  inline.
- We deliberately avoid `geopandas` + `contextily` here. They are
  excellent tools for publication-quality static maps with
  reprojected basemaps, but heavier to install (GDAL, GEOS, PROJ)
  and not needed for an interactive web map. If you want static
  maps with basemaps, they are the right choice.

## Step 1 — Defining the SPARQL query

The query asks the N4O KG for every `oghamonto:OghamSite`, its
label, its GeoSPARQL geometry, the `County` it lies in, and a count
of catalogued Ogham stones (`oghamonto:OghamStone_CIIC`) disclosed
at that site. We group by site — one row per site with its
aggregated stone count.

Two notes on query design that generalise to other knowledge-graph
queries:

- **`geo:hasGeometry/geo:asWKT`** uses SPARQL property-path
  notation to traverse two predicates in a single triple pattern.
  It is equivalent to introducing an intermediate variable, but
  shorter.
- **`COUNT(DISTINCT ?stone)`** — not just `COUNT(?stone)`. A stone
  could in principle be linked through more than one path;
  `DISTINCT` makes sure we count each stone once.

In [ ]:
import re
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import folium

SPARQL_ENDPOINT = "https://graph.nfdi4objects.net/api/sparql"

SPARQL_QUERY = """
PREFIX oghamonto: <http://ontology.ogham.link/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
SELECT ?item ?label ?geo ?county (count(distinct ?stone) as ?count) WHERE {
  ?item <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ontology.ogham.link/OghamSite> .
  ?item rdfs:label ?label .
  ?item geo:hasGeometry/geo:asWKT ?geo .
  ?item oghamonto:within ?c .
  ?c a oghamonto:County .
  ?c rdfs:label ?county .
  ?stone oghamonto:disclosedAt ?item .
  ?stone a oghamonto:OghamStone_CIIC .
} GROUP BY ?item ?label ?geo ?county ORDER BY DESC(?count)
"""

def fetch_sparql(query, endpoint=SPARQL_ENDPOINT):
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.queryAndConvert()["results"]["bindings"]

# Case-insensitive WKT parser: handles both 'Point(...)' (Wikidata) and
# 'POINT(...)' (N4O KG) and variations; also tolerates an optional SRID
# prefix like '<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(...)'.
_WKT_POINT_RE = re.compile(r"point\s*\(\s*([-+\d.eE]+)\s+([-+\d.eE]+)\s*\)",
                           re.IGNORECASE)

def parse_wkt_point(wkt):
    """Parse a WKT Point literal 'POINT(lon lat)' (any case) into (lat, lon)."""
    if not wkt:
        return (None, None)
    m = _WKT_POINT_RE.search(wkt)
    if not m:
        return (None, None)
    try:
        lon, lat = float(m.group(1)), float(m.group(2))
        return (lat, lon)
    except ValueError:
        return (None, None)

print("✓ Helpers defined.")

## Step 2 — Loading the data

We flatten the SPARQL bindings into plain records, parse each WKT
point into `(latitude, longitude)` pairs, and drop any rows without
coordinates before building the DataFrame.

In [ ]:
results = fetch_sparql(SPARQL_QUERY)

rows = []
for r in results:
    lat, lon = parse_wkt_point(r.get("geo", {}).get("value"))
    rows.append({
        "item":      r["item"]["value"],
        "label":     r["label"]["value"],
        "county":    r["county"]["value"],
        "count":     int(r["count"]["value"]),
        "latitude":  lat,
        "longitude": lon,
    })

df = pd.DataFrame(rows).dropna(subset=["latitude", "longitude"])
assert not df.empty, "⚠ No records with coordinates returned — check query."
print(f"✓ {len(df)} Ogham sites loaded across {df['county'].nunique()} counties "
      f"(total stones: {df['count'].sum()}).")
df.head()

## Step 3a — Map with county-coloured markers

We give each county a distinctive colour from a 20-colour palette,
then add one `CircleMarker` per site. `folium` renders the map
inline as an HTML + JavaScript widget.

In [ ]:
assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

palette = [
    "#e6194B", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#469990", "#9A6324", "#800000",
    "#808000", "#000075", "#e6beff", "#fabed4", "#aaffc3",
    "#fffac8", "#dcbeff", "#ffd8b1", "#a9a9a9", "#ffe119",
]
counties = sorted(df["county"].unique())
county_colors = {c: palette[i % len(palette)] for i, c in enumerate(counties)}

# Centre the map on the mean of all coordinates
m = folium.Map(
    location=[df["latitude"].mean(), df["longitude"].mean()],
    zoom_start=7,
    tiles="OpenStreetMap",
)

# One FeatureGroup per county — makes each toggleable in the layer control
county_groups = {
    c: folium.FeatureGroup(name=f"{c} ({(df['county'] == c).sum()})")
    for c in counties
}

for _, row in df.iterrows():
    popup_html = (
        f"<b>{row['label']}</b><br>"
        f"County: {row['county']}<br>"
        f"Ogham stones: {row['count']}<br>"
        f'<a href="{row["item"]}" target="_blank">Site URI &#8599;</a>'
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=county_colors[row["county"]],
        weight=1.5,
        fill=True,
        fill_color=county_colors[row["county"]],
        fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(county_groups[row["county"]])

for g in county_groups.values():
    g.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

## Step 3b — Map styled by stone count

The second map shows the *same* sites as Step 3a, but here the
stone count per site drives the visual encoding: each marker's
**radius is proportional to the count**, and its **colour follows
a continuous `viridis` gradient** from low (dark purple) to high
(bright yellow). The two encodings reinforce each other — busy
sites stand out clearly.

A design note on why this is not a heatmap: heatmaps are best at
answering *"where are points concentrated in space?"*, not
*"where are the large values?"*. Since our dataset has
well-separated sites with very uneven counts (one site with 30+
stones, many with just one or two), a density heatmap weighted
by count drowns the low-count sites in a hazy background and
makes the high-count ones barely distinguishable. Proportional
circles are the honest visualisation for data like this.

We use `matplotlib`'s colormap machinery to convert each count to
a hex colour, which `folium.CircleMarker` understands directly.

In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex

assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

max_count = int(df["count"].max())
cmap = plt.get_cmap("viridis")

# Discretise counts into N bins with equal-width ranges starting at 1.
# Markers and legend share the same bin_color() function so the legend
# always matches what's on the map.
N_BINS = 5
step = math.ceil(max_count / N_BINS)

def bin_index(count):
    return min((count - 1) // step, N_BINS - 1)

def bin_range(idx):
    lo = idx * step + 1
    hi = min((idx + 1) * step, max_count)
    return lo, hi

def bin_color(idx):
    # Colour from the *middle* of each bin's share of the viridis
    # gradient, so bins are clearly different but not at the extremes.
    return to_hex(cmap((idx + 0.5) / N_BINS))

def radius_for(count):
    # sqrt scaling so area (not radius) is proportional to count —
    # perceptually this is what the eye judges. 5 px floor keeps the
    # smallest site clickable; 22 px cap prevents the biggest from
    # swamping the map.
    return 5 + 17 * math.sqrt(count / max_count)

m = folium.Map(
    location=[df["latitude"].mean(), df["longitude"].mean()],
    zoom_start=7,
    tiles="OpenStreetMap",
)

for _, row in df.iterrows():
    popup_html = (
        f"<b>{row['label']}</b><br>"
        f"County: {row['county']}<br>"
        f"Ogham stones: <b>{row['count']}</b><br>"
        f'<a href="{row["item"]}" target="_blank">Site URI &#8599;</a>'
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=radius_for(row["count"]),
        color="#333",
        weight=1,
        fill=True,
        fill_color=bin_color(bin_index(row["count"])),
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(m)

# Discrete colour-scale legend in the bottom-right as raw HTML.
# folium.Map exposes the underlying Branch HTML root for free-form
# additions like this; the legend is positioned with plain CSS.
legend_rows = []
for i in range(N_BINS - 1, -1, -1):  # highest bin first
    lo, hi = bin_range(i)
    label = f"{lo}" if lo == hi else f"{lo}–{hi}"
    legend_rows.append(
        f'<div><span style="display:inline-block;width:14px;height:14px;'
        f'background:{bin_color(i)};border:1px solid #333;'
        f'border-radius:50%;margin-right:6px;vertical-align:middle"></span>'
        f'{label}</div>'
    )
legend_html = (
    '<div style="position:fixed;bottom:30px;right:30px;z-index:9999;'
    'background:rgba(255,255,255,0.92);padding:8px 12px;border-radius:4px;'
    'font:12px/1.5 system-ui,sans-serif;box-shadow:0 1px 4px rgba(0,0,0,0.2)">'
    '<b>Stones per site</b><br>' + "".join(legend_rows) + '</div>'
)
m.get_root().html.add_child(folium.Element(legend_html))
m

## Step 4 — Exploring the data

The cells below are a free playground — filter the DataFrame by
county, compute per-county aggregates, or extend the query
yourself. Remember: in this notebook *one row = one site*, with a
`count` column holding the number of Ogham stones disclosed there.

In [ ]:
assert 'df' in dir(), "⚠ Please run the data-loading cell first."

# Example: list every Ogham site from a given county, with coordinates
# and stone counts. Change the filter below to explore other counties.
county_filter = "Kerry"

subset = (
    df[df["county"] == county_filter]
      [["label", "count", "latitude", "longitude"]]
      .sort_values("count", ascending=False)
      .reset_index(drop=True)
)
print(f"Ogham sites in {county_filter}: {len(subset)} "
      f"(total stones: {subset['count'].sum()})")
subset

In [ ]:
assert 'df' in dir(), "⚠ Please run the data-loading cell first."

# Per-county summary: number of sites, total stone count, and the
# geographic centroid of the sites.
summary = (
    df.groupby("county")
      .agg(
          n_sites=("item", "nunique"),
          n_stones=("count", "sum"),
          mean_lat=("latitude", "mean"),
          mean_lon=("longitude", "mean"),
      )
      .sort_values("n_stones", ascending=False)
)
summary

---

*This notebook is part of the Open Educational Resources of
[NFDI4Objects](https://www.nfdi4objects.net/).*